## **Laboratorio: Text-to-SQL con Vanna.ai**

**Text-to-SQL** es la tarea de traducir una pregunta en lenguaje natural a una consulta SQL válida. Vanna.ai combina dos tecnologías para hacerlo de forma fiable:

1. **RAG (Retrieval-Augmented Generation):** Antes de generar SQL, Vanna recupera del vector store los ejemplos, definiciones y documentación más relevantes para la pregunta.
2. **LLM:** Con ese contexto, el modelo de lenguaje (GPT-4o-mini) genera la consulta SQL.

```
Pregunta  →  Vanna (RAG + LLM)  →  SQL  →  SQLite  →  DataFrame resultado
```

---
> ⚠️ Las celdas marcadas con **📝 COMPLETAR** tienen huecos que debes rellenar antes de ejecutarlas.

**Prerequisito:** Clave de API de OpenAI. Configúrala en la terminal antes de abrir Jupyter:
```bash
export OPENAI_API_KEY="sk-..."
```

---
## Sección 1: Preparación del Entorno

**Celda 1: Instalación**

In [ ]:
!pip install "vanna[chromadb,openai]" seaborn pandas plotly streamlit

**Celda 2: Imports y API Key**

In [ ]:
import seaborn as sns
import pandas as pd
import sqlite3
import os

from vanna.openai import OpenAI_Chat
from vanna.chromadb import ChromaDB_VectorStore

# ── Introduce aquí tu clave de API de OpenAI ──────────────────────────────
OPENAI_API_KEY = "sk-..."   # ← Reemplaza con tu clave real
# ──────────────────────────────────────────────────────────────────────────

if OPENAI_API_KEY == "sk-..." or not OPENAI_API_KEY:
    print("⚠️  Recuerda sustituir 'sk-...' por tu clave real antes de continuar.")
else:
    print(f"✅ API Key configurada: {OPENAI_API_KEY[:8]}...")

---
## Sección 2: Crear la Base de Datos SQLite

SQLite es una base de datos relacional embebida en un único archivo — perfecta para prototipos. `pandas` permite guardar un DataFrame como tabla SQL con `df.to_sql()`.

**Celda 3: Preparar el Dataset** *(ya completada)*

In [ ]:
# Cargar Palmer Penguins y traducir columnas al español
df = sns.load_dataset('penguins').dropna()
df = df.rename(columns={
    'species': 'especie', 'island': 'isla',
    'bill_length_mm': 'longitud_pico_mm', 'bill_depth_mm': 'profundidad_pico_mm',
    'flipper_length_mm': 'longitud_aleta_mm', 'body_mass_g': 'masa_corporal_g',
    'sex': 'sexo'
})
df['sexo'] = df['sexo'].map({'male': 'macho', 'female': 'hembra'})

print(f"Dataset: {len(df)} filas")
print(f"Columnas: {list(df.columns)}")
df.head()

**Celda 4: Guardar en SQLite** — 📝 COMPLETAR

In [ ]:
DB_PATH = 'pinguinos.db'

# 📝 COMPLETAR: guarda el DataFrame en SQLite
# Pasos:
#   1. conn = sqlite3.connect(DB_PATH)
#   2. df.to_sql('pinguinos', conn, if_exists='replace', index=False)
#   3. conn.close()
### EMPIEZA TU CÓDIGO ###
# TU CÓDIGO AQUÍ

raise NotImplementedError("Completa los 3 pasos antes de continuar")
### FIN DE TU CÓDIGO ###

print(f"✅ Base de datos creada: {DB_PATH} ({os.path.getsize(DB_PATH)/1024:.1f} KB)")


**Celda 5: Explorar la Base de Datos** *(ejecuta y observa el esquema)*

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Ver el esquema de la tabla
esquema = pd.read_sql("PRAGMA table_info(pinguinos)", conn)
print("=== Esquema de la tabla 'pinguinos' ===")
print(esquema[['name', 'type']].to_string(index=False))

# Primera consulta SQL manual
print("\n=== Conteo por especie ===")
resultado = pd.read_sql(
    "SELECT especie, COUNT(*) as total FROM pinguinos GROUP BY especie ORDER BY total DESC",
    conn
)
print(resultado.to_string(index=False))
conn.close()

---
## Sección 3: Configurar Vanna *(ya completada)*

Vanna usa herencia múltiple de Python para combinar un **vector store** (ChromaDB, almacena el contexto) con un **LLM** (OpenAI, genera el SQL).

**Celda 6: Instanciar Vanna**

In [ ]:
# Definir la clase combinando ChromaDB + OpenAI
class MiVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

# Instanciar con GPT-4o-mini (barato y suficiente para SQL)
vn = MiVanna(config={
    'api_key': OPENAI_API_KEY,
    'model':   'gpt-4o-mini'
})

# Conectar a la base de datos
vn.connect_to_sqlite(DB_PATH)
print("✅ Vanna configurado y conectado.")

---
## Sección 4: Entrenar a Vanna

"Entrenar" a Vanna significa **poblar el vector store** con contexto sobre tu base de datos. Hay tres tipos:

| Tipo | Función | Qué aporta |
|------|---------|------------|
| DDL | `vn.train(ddl=...)` | Esquema de tablas y columnas |
| Documentación | `vn.train(documentation=...)` | Descripción en lenguaje natural |
| Ejemplos SQL | `vn.train(sql=...)` | Consultas de referencia |

Cuanto más rico sea el contexto, más precisas serán las consultas generadas.

**Celda 7: Entrenamiento con DDL** — 📝 COMPLETAR

In [ ]:
# 📝 COMPLETAR: escribe el DDL de la tabla 'pinguinos'
# Mira el esquema de la Celda 5 y completa:
#   - El tipo de cada columna: TEXT (texto) o REAL (número decimal)
#   - Un comentario SQL  -- describiendo cada columna
#   - Los valores posibles para especie, isla y sexo
#
# Ejemplo de una columna bien hecha:
#   especie TEXT,  -- Especie: 'Adelie', 'Chinstrap' o 'Gentoo'

### EMPIEZA TU CÓDIGO ###
vn.train(ddl="""
    CREATE TABLE pinguinos (
        especie              # TU CÓDIGO AQUÍ,
        isla                 # TU CÓDIGO AQUÍ,
        longitud_pico_mm     # TU CÓDIGO AQUÍ,
        profundidad_pico_mm  # TU CÓDIGO AQUÍ,
        longitud_aleta_mm    # TU CÓDIGO AQUÍ,
        masa_corporal_g      # TU CÓDIGO AQUÍ,
        sexo                 # TU CÓDIGO AQUÍ
    );
""")
### FIN DE TU CÓDIGO ###

print("✅ DDL añadido al vector store.")


**Celda 8: Entrenamiento con Documentación** — 📝 COMPLETAR

In [ ]:
# 📝 COMPLETAR: escribe la documentación del dataset en lenguaje natural
# Debe incluir al menos:
#   1. Cuántos registros tiene y qué representan
#   2. Los valores posibles de especie, isla y sexo
#   3. Un dato relevante del dominio

### EMPIEZA TU CÓDIGO ###
vn.train(documentation="""
# TU CÓDIGO AQUÍ
""")
### FIN DE TU CÓDIGO ###

# Segundo bloque — ya dado
vn.train(documentation="""
Relaciones conocidas en el dataset:
- La especie Chinstrap solo se encuentra en la isla Dream.
- La especie Gentoo solo se encuentra en la isla Biscoe.
- La especie Adelie se encuentra en las tres islas.
- Existe una correlación positiva fuerte entre longitud_aleta_mm y masa_corporal_g.
""")

print("✅ Documentación añadida al vector store.")


**Celda 9: Entrenamiento con Ejemplos SQL** — 📝 COMPLETAR

In [ ]:
# Ejemplos dados — no los modifiques
ejemplos_dados = [
    "SELECT especie, COUNT(*) AS total FROM pinguinos GROUP BY especie ORDER BY total DESC",
    "SELECT isla, especie, COUNT(*) AS total FROM pinguinos GROUP BY isla, especie ORDER BY isla",
    "SELECT especie, ROUND(AVG(masa_corporal_g), 1) AS masa_media FROM pinguinos GROUP BY especie ORDER BY masa_media DESC",
    "SELECT especie, MIN(masa_corporal_g) AS minimo, MAX(masa_corporal_g) AS maximo FROM pinguinos GROUP BY especie",
    "SELECT * FROM pinguinos WHERE especie = 'Gentoo' ORDER BY masa_corporal_g DESC LIMIT 10",
]
for sql in ejemplos_dados:
    vn.train(sql=sql)
print(f"✅ {len(ejemplos_dados)} ejemplos dados añadidos.")

# 📝 COMPLETAR: escribe 5 ejemplos SQL propios
# Cubre un tipo distinto en cada uno:
#   1. WHERE por sexo
#   2. GROUP BY dos columnas
#   3. ORDER BY + LIMIT (top N)
#   4. COUNT con WHERE numérico
#   5. AVG de varias columnas GROUP BY isla
### EMPIEZA TU CÓDIGO ###
mis_ejemplos = [
    # TU CÓDIGO AQUÍ — 5 strings SQL
]
### FIN DE TU CÓDIGO ###

for sql in mis_ejemplos:
    vn.train(sql=sql)
print(f"✅ {len(mis_ejemplos)} ejemplos propios añadidos.")


**Celda 10: Verificar el Vector Store**

In [ ]:
datos_entrenamiento = vn.get_training_data()
print(f"Total de elementos en el vector store: {len(datos_entrenamiento)}")
print("\nTipos:")
print(datos_entrenamiento['training_data_type'].value_counts().to_string())

---
## Sección 5: Consultas en Lenguaje Natural

Con Vanna entrenado, puedes usar estos métodos:
- `vn.generate_sql(pregunta)` — genera el SQL sin ejecutarlo
- `vn.run_sql(sql)` — ejecuta el SQL y devuelve un DataFrame

**Celda 11: Función de Consulta** — 📝 COMPLETAR

In [ ]:
# 📝 COMPLETAR: implementa la función consultar()
# La función debe:
#   1. Llamar a vn.generate_sql(pregunta) → genera el SQL
#   2. Imprimir el SQL generado
#   3. Llamar a vn.run_sql(sql) → ejecuta y devuelve un DataFrame
#   4. Mostrar el resultado con display(resultado)
#   5. Devolver el resultado

def consultar(pregunta):
    print(f"❓ Pregunta: {pregunta}")
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Implementa el cuerpo de la función")
    ### FIN DE TU CÓDIGO ###


**Celda 12: Prueba las Consultas** — 📝 COMPLETAR

In [ ]:
# Consulta 1 — ya dada
consultar("¿Cuántos pingüinos hay de cada especie?")

In [ ]:
# Consulta 2 — ya dada
consultar("¿Cuál es la masa corporal media de cada especie ordenada de mayor a menor?")

In [ ]:
# 📝 COMPLETAR: escribe 3 preguntas propias en lenguaje natural
# Deben ser distintas a las anteriores
### EMPIEZA TU CÓDIGO ###
consultar("# TU PREGUNTA AQUÍ")   # Pregunta 3
### FIN DE TU CÓDIGO ###


In [ ]:
### EMPIEZA TU CÓDIGO ###
consultar("# TU PREGUNTA AQUÍ")   # Pregunta 4
### FIN DE TU CÓDIGO ###


In [ ]:
### EMPIEZA TU CÓDIGO ###
consultar("# TU PREGUNTA AQUÍ")   # Pregunta 5
### FIN DE TU CÓDIGO ###


**Celda 13: Visualizar un Resultado** — 📝 COMPLETAR

In [ ]:
import plotly.express as px

df_resultado = consultar("¿Cuál es la masa corporal media por especie y sexo?")

# 📝 COMPLETAR: crea un gráfico de barras con px.bar()
# Parámetros: x (especie), y (masa_media), color (sexo), barmode ('group')
### EMPIEZA TU CÓDIGO ###
fig = px.bar(
    df_resultado,
    # TU CÓDIGO AQUÍ — rellena x, y, color y barmode
    title="Masa corporal media por especie y sexo"
)
fig.show()
### FIN DE TU CÓDIGO ###


---
## Sección 6: Chatbot de SQL con Streamlit *(código dado — ejecútalo y pruébalo)*

Esta sección usa lo aprendido en el Lab de Streamlit para construir una interfaz de chat. El código está completo — tu tarea es **ejecutarlo, entenderlo y probarlo**.

Componentes nuevos de Streamlit que aparecen aquí:
| Componente | Uso |
|-----------|-----|
| `@st.cache_resource` | Caché para objetos pesados (modelos, conexiones) — no serializa |
| `st.chat_message(role)` | Burbujas de chat (user / assistant) |
| `st.chat_input(texto)` | Cuadro de texto en la parte inferior |
| `st.spinner(texto)` | Indicador de carga durante operaciones lentas |
| `st.rerun()` | Fuerza un rerun inmediato del script |

**Celda 14: Crear el Chatbot** *(ejecuta sin modificar)*

In [ ]:
%%writefile chatbot_sql.py
import streamlit as st
import pandas as pd
import plotly.express as px
from vanna.openai import OpenAI_Chat
from vanna.chromadb import ChromaDB_VectorStore

st.set_page_config(page_title="Asistente SQL de Pingüinos", page_icon="🐧", layout="wide")

class MiVanna(ChromaDB_VectorStore, OpenAI_Chat):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        OpenAI_Chat.__init__(self, config=config)

# @st.cache_resource: para objetos pesados que no se pueden serializar
@st.cache_resource
def inicializar_vanna(api_key):
    vn = MiVanna(config={'api_key': api_key, 'model': 'gpt-4o-mini'})
    vn.connect_to_sqlite('pinguinos.db')
    return vn

api_key_input = st.sidebar.text_input(
    "🔑 OpenAI API Key",
    type="password",
    placeholder="sk-..."
)
if not api_key_input:
    st.info("👈 Introduce tu OpenAI API Key en el panel lateral para comenzar.")
    st.stop()

vn = inicializar_vanna(api_key_input)

# --- Encabezado ---
st.title("🐧 Asistente SQL de Pingüinos")
st.markdown("Pregunta en español sobre el dataset y la IA traducirá tu pregunta a SQL automáticamente.")

# --- Sidebar con ejemplos ---
st.sidebar.header("💡 Preguntas de ejemplo")
for p in [
    "¿Cuántos pingüinos hay de cada especie?",
    "¿Cuál es la masa media por especie?",
    "¿Qué especie tiene el pico más largo en promedio?",
    "¿Cuántos pingüinos hay en cada isla por especie?",
    "Muestra los 5 pingüinos más pesados con su especie.",
]:
    st.sidebar.markdown(f"• *{p}*")

# --- Historial del chat con session_state ---
# session_state persiste entre reruns — imprescindible para el historial de chat
if 'mensajes' not in st.session_state:
    st.session_state.mensajes = [{
        'role': 'assistant',
        'content': '¡Hola! Hazme cualquier pregunta sobre los pingüinos del Archipiélago Palmer.',
        'df': None, 'sql': None
    }]

# Renderizar historial
for msg in st.session_state.mensajes:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])
        if msg['sql']:
            with st.expander("🔍 Ver SQL generado"):
                st.code(msg['sql'], language='sql')
        if msg['df'] is not None and not msg['df'].empty:
            st.dataframe(msg['df'], use_container_width=True)
            cols = msg['df'].columns.tolist()
            if len(cols) == 2 and pd.api.types.is_numeric_dtype(msg['df'][cols[1]]):
                fig = px.bar(msg['df'], x=cols[0], y=cols[1])
                st.plotly_chart(fig, use_container_width=True)

# --- Input del usuario ---
pregunta = st.chat_input("Escribe tu pregunta sobre los pingüinos...")

if pregunta:
    st.session_state.mensajes.append({'role': 'user', 'content': pregunta, 'df': None, 'sql': None})
    with st.chat_message('user'):
        st.markdown(pregunta)

    with st.chat_message('assistant'):
        with st.spinner('Generando SQL...'):
            try:
                sql_generado = vn.generate_sql(pregunta)
                df_resultado = vn.run_sql(sql_generado)
                respuesta = f"He encontrado **{len(df_resultado)} resultado(s)**:" if df_resultado is not None and not df_resultado.empty else "La consulta no devolvió resultados."
            except Exception as e:
                respuesta = f"❌ No he podido procesar esa pregunta. Intenta reformularla.\nError: {str(e)}"
                sql_generado = None
                df_resultado = None

        st.markdown(respuesta)
        if sql_generado:
            with st.expander("🔍 Ver SQL generado"):
                st.code(sql_generado, language='sql')
        if df_resultado is not None and not df_resultado.empty:
            st.dataframe(df_resultado, use_container_width=True)
            cols = df_resultado.columns.tolist()
            if len(cols) == 2 and pd.api.types.is_numeric_dtype(df_resultado[cols[1]]):
                fig = px.bar(df_resultado, x=cols[0], y=cols[1], title=pregunta)
                st.plotly_chart(fig, use_container_width=True)

        st.session_state.mensajes.append({
            'role': 'assistant', 'content': respuesta,
            'sql': sql_generado, 'df': df_resultado
        })

# Botón para limpiar el chat
st.sidebar.divider()
if st.sidebar.button("🗑️ Limpiar conversación"):
    st.session_state.mensajes = [{'role': 'assistant', 'content': '¡Conversación reiniciada!', 'df': None, 'sql': None}]
    st.rerun()

### 🔁 Checkpoint — Ejecutar el Chatbot

```bash
streamlit run chatbot_sql.py
```

**Prueba estas interacciones:**
1. Escribe una de las preguntas de ejemplo del sidebar.
2. Haz clic en "Ver SQL generado" para ver qué SQL produjo Vanna.
3. Prueba una pregunta más compleja que no esté en los ejemplos.
4. Haz 3-4 preguntas seguidas — el historial debe mantenerse.
5. Limpia la conversación con el botón del sidebar.

---
## Sección 7: Mejorar a Vanna cuando Falla

Si Vanna genera SQL incorrecto, añade el SQL correcto como ejemplo y mejorará.

**Celda 15: Añadir Ejemplos de Mejora** — 📝 COMPLETAR

In [ ]:
# 📝 COMPLETAR: prueba el chatbot, encuentra una pregunta donde Vanna
# genere SQL incorrecto y añade el SQL correcto al vector store

# Pregunta problemática que encontraste:
# pregunta_problematica = "..."

### EMPIEZA TU CÓDIGO ###
sql_correcto = """
# TU CÓDIGO AQUÍ — SQL correcto para esa pregunta
"""
# vn.train(sql=sql_correcto)
# print("✅ Ejemplo de mejora añadido")
### FIN DE TU CÓDIGO ###

datos = vn.get_training_data()
print(f"Total en vector store: {len(datos)} elementos")


---
## Ejercicio Extra (opcional)

Si terminas antes, intenta **una** de estas mejoras:

1. **Segunda tabla:** Crea una tabla `medias_especie` con las medias de cada especie usando pandas y `to_sql()`. Entrena a Vanna con su DDL.

2. **Botón de descarga:** Añade al chatbot un `st.download_button` que aparezca después de cada respuesta para descargar el DataFrame como CSV.

3. **Vanna con Ollama (sin coste):** Cambia el LLM a Ollama local:
   ```python
   from vanna.ollama import Ollama
   class MiVannaLocal(ChromaDB_VectorStore, Ollama):
       ...
   vn = MiVannaLocal(config={'ollama_host': 'http://localhost:11434', 'model': 'llama3.1'})
   ```

---
## Conclusiones

En este laboratorio has aprendido:

| Concepto | Descripción |
|---------|-------------|
| **Text-to-SQL** | Traducción de lenguaje natural a SQL con LLMs |
| **RAG** | Recuperar contexto relevante antes de generar |
| **SQLite** | Base de datos local en un archivo |
| **`df.to_sql()`** | Pasar datos de pandas a SQL |
| **Entrenamiento Vanna** | DDL + Documentación + Ejemplos SQL |
| **`st.chat_message`** | Interfaz de chat en Streamlit |
| **`st.cache_resource`** | Caché para modelos y conexiones |
| **`st.session_state`** | Historial persistente entre reruns |

**🎉 ¡Felicidades! Has construido un chatbot de SQL con Inteligencia Artificial.**